# Задание 6. Создание прогнозатора категорий

Классификатор на основе TF-IDF и мультиномиального наивного байесовского классификатора. Обучение на датасете 20 Newsgroups, пять категорий: хоккей и бейсбол (спорт), оружие и Ближний Восток (политика), космос (наука).

In [11]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer

In [12]:
# Ключи категорий - имена групп из scikit-learn
categories = ["rec.sport.hockey", "rec.sport.baseball", "talk.politics.guns",
              "talk.politics.mideast", "sci.space"]

training_data = fetch_20newsgroups(subset="train", categories=categories,
                                   shuffle=True, random_state=7)
print("Документов:", len(training_data.data))
print("Категории:", training_data.target_names)

Документов: 2900
Категории: ['rec.sport.baseball', 'rec.sport.hockey', 'sci.space', 'talk.politics.guns', 'talk.politics.mideast']


In [13]:
# Счетчики слов -> TF-IDF
count_vectorizer = CountVectorizer()
train_counts = count_vectorizer.fit_transform(training_data.data)

tfidf = TfidfTransformer()
train_tfidf = tfidf.fit_transform(train_counts)

classifier = MultinomialNB().fit(train_tfidf, training_data.target)

In [14]:
# Тестовые предложения, написаны вручную и не входят в обучающую выборку
test_sentences = [
    "The goalie made three saves in the third period and the crowd went wild.",
    "The senator argued that the new bill would restrict who can buy a rifle.",
    "The rocket will launch from the cape next week carrying a satellite into orbit.",
    "Negotiations over the border settlements stalled again after the latest summit.",
    "The pitcher struck out nine batters before the bullpen took over in the eighth.",
]

test_counts = count_vectorizer.transform(test_sentences)
test_tfidf = tfidf.transform(test_counts)

In [15]:
predictions = classifier.predict(test_tfidf)

for sentence, pred in zip(test_sentences, predictions):
    print(training_data.target_names[pred], "|", sentence)

rec.sport.hockey | The goalie made three saves in the third period and the crowd went wild.
talk.politics.guns | The senator argued that the new bill would restrict who can buy a rifle.
sci.space | The rocket will launch from the cape next week carrying a satellite into orbit.
talk.politics.mideast | Negotiations over the border settlements stalled again after the latest summit.
rec.sport.baseball | The pitcher struck out nine batters before the bullpen took over in the eighth.


In [16]:
# Проверка вклада слов в решение классификатора для каждого тестового предложения
# вклад = TF-IDF вес * (log P(слово|предсказанный класс) - среднее log P(слово|другие классы))
# dP = изменение вероятности предсказанного класса при удалении слова из предложения
logp = classifier.feature_log_prob_
vocab = count_vectorizer.vocabulary_

for sentence in test_sentences:
    x = tfidf.transform(count_vectorizer.transform([sentence]))
    pred = classifier.predict(x)[0]
    base = classifier.predict_proba(x)[0][pred]
    others = [c for c in range(len(training_data.target_names)) if c != pred]

    print(training_data.target_names[pred], "|", sentence)
    rows = []
    for w in set(sentence.lower().replace(".", "").replace(",", "").replace("?", "").split()):
        if w not in vocab:
            continue
        j = vocab[w]
        contrib = x[0, j] * (logp[pred, j] - logp[others, j].mean())
        x2 = x.copy()
        x2[0, j] = 0
        dp = classifier.predict_proba(x2)[0][pred] - base
        rows.append((contrib, w, dp))
    for contrib, w, dp in sorted(rows, reverse=True)[:6]:
        print(f"   {w:<12} вклад={contrib:+.3f}  без слова dP={dp:+.3f}")
    print()

rec.sport.hockey | The goalie made three saves in the third period and the crowd went wild.
   goalie       вклад=+0.563  без слова dP=-0.136
   period       вклад=+0.356  без слова dP=-0.090
   saves        вклад=+0.234  без слова dP=-0.052
   third        вклад=+0.098  без слова dP=-0.019
   went         вклад=+0.094  без слова dP=-0.021
   made         вклад=+0.035  без слова dP=-0.009

talk.politics.guns | The senator argued that the new bill would restrict who can buy a rifle.
   rifle        вклад=+0.372  без слова dP=-0.092
   buy          вклад=+0.311  без слова dP=-0.077
   bill         вклад=+0.258  без слова dP=-0.062
   restrict     вклад=+0.196  без слова dP=-0.049
   senator      вклад=+0.180  без слова dP=-0.044
   argued       вклад=+0.160  без слова dP=-0.040

sci.space | The rocket will launch from the cape next week carrying a satellite into orbit.
   orbit        вклад=+0.735  без слова dP=-0.174
   launch       вклад=+0.696  без слова dP=-0.164
   satellite    вкла